In [ ]:
# chemRxiv: "/10.26434/"
# preprints.org: "/10.20944/"
# e.g.
# doi.org/10.20944/preprints202512.2746.v1
# doi.org/10.26434/chemrxiv-2026-zgmgw/v2
# doi.org/10.26434/chemrxiv-2022-lf9dc

In [ ]:
# doi.org/10.20944/preprints202601.2387.v1
# doi.org/10.20944/preprints202601.2387.v1


In [25]:
from constant import openalex_path
import polars as pl
from constant import data_path

doi_lf = (
    pl.scan_parquet(f"{openalex_path}/works/doi/*")
    .with_columns(
        pl.col("doi")
        .str.to_lowercase()
        .str.replace_all(r"https://doi.org/", "", literal=True)
    )
    .drop_nulls()
)
# source_lf = pl.scan_parquet(f"{openalex_path}/works/primary_location_id/*")
ref_lf = pl.scan_parquet(f"{openalex_path}/works/referenced_works/*")

In [ ]:
doi_lf.filter(
    pl.col("doi").str.starts_with("10.20944/")
).sink_ndjson("/home/ec2-user/r2/preprintsorg/preprintsorg_preprints.dedup_openalex.jsonl")

doi_lf.filter(
    pl.col("doi").str.starts_with("10.26434/")
).sink_ndjson("/home/ec2-user/r2/chemrxiv/chemrxiv_preprints.dedup_openalex.jsonl")

/tmp/ipykernel_5572/4214481721.py:3: UserWarning: '(default_)region' not set; polars will try to get it from bucket

Set the region manually to silence this warning.
  ).sink_ndjson("/home/ec2-user/r2/preprintsorg/preprintsorg_preprints.dedup_openalex.jsonl")


In [24]:
preprintsorg = (
    pl.scan_ndjson("/home/ec2-user/r2/preprintsorg/preprintsorg_preprints.dedup_openalex.jsonl")
    # .with_columns(
    #     pl.col("doi").str.slice(0, 30)
    # )
    # .unique("doi")
)
chemrxiv = (
    pl.scan_ndjson("/home/ec2-user/r2/chemrxiv/chemrxiv_preprints.dedup_openalex.jsonl")
    # .with_columns(
    #     pl.col("doi")
    #     .str.to_lowercase()
    #     .str.replace(r"(\.v\d+|-v\d+)$", "")
    # )
    # .unique("doi")
)

In [ ]:
preprintsorg.count().collect(engine='streaming')
# chemrxiv.count().collect(engine='streaming')


work_id,doi
u32,u32
87491,87491


In [16]:
len("/10.26434/chemrxiv-2022-jjm0j")

29

In [21]:
import polars as pl

normalized = (
    chemrxiv
    .with_columns(
        pl.col("doi")
        .str.to_lowercase()
        .str.replace(r"(\.v\d+|-v\d+)$", "")
        .alias("doi_base")
    )
)

normalized.select(
    pl.n_unique("doi_base")
).collect(engine="streaming")


doi_base
u32
31233


In [27]:
(
    ref_lf
    .join(chemrxiv, on='work_id', how='anti')
    .explode("referenced_works")
    .join(chemrxiv, left_on='referenced_works', right_on='work_id', how='right')
    .head()
    .drop_nulls("work_id")
    # .count()
    .collect(engine='streaming')
)


work_id,work_id_right,doi
str,str,str
"""W4406864317""","""W4282832171""","""/10.26434/chemrxiv-2022-8v4h3"""
"""W4407027432""","""W4388568409""","""/10.26434/chemrxiv-2023-tp76q"""
"""W4407219015""","""W4321094173""","""/10.26434/chemrxiv-2023-p6bg9"""
"""W4407219015""","""W4388454605""","""/10.26434/chemrxiv-2023-x25jj"""
"""W4407753378""","""W4241128276""","""/10.26434/chemrxiv.13378148.v1"""


In [ ]:
(
    preprintsorg
    .collect(engine='streaming')
    .write_csv("data/preprintsorg_count.csv")
)

In [29]:
(
    normalized
    .collect(engine='streaming')
    .write_csv("data/chemrxiv_count.csv")
)

In [ ]:
import time
import requests

BASE = "https://api.crossref.org/v1/works"
MAILTO = "miura-tchiaki873@g.ecc.u-tokyo.ac.jp"

params = {
    "filter": "type:posted-content,prefix:10.26434",
    "rows": 10,
    "cursor": "*",
    "mailto": MAILTO,
}

sess = requests.Session()
headers = {
    # 推奨: 自分のUAを名乗る（mailto は query に入れているので必須ではないが、丁寧）
    "User-Agent": f"my-harvester/0.1 (mailto:{MAILTO})"
}

seen = 0
while True:
    r = sess.get(BASE, params=params, headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json()["message"]

    items = data.get("items", [])
    if not items:
        break

    # ここで items をJSONLに書くなど（例: doi, title, author, posted, relation等）
    for it in items:
        

    seen += len(items)
    params["cursor"] = data["next-cursor"]

    # polite pool のレート制限/同時接続制限に配慮してスリープ
    time.sleep(0.2)

print("retrieved:", seen)